假设清算阈值不会变，没有根据健康因子的调整去求出解对币的价值进行解耦

In [7]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime, timezone
from decimal import Decimal, getcontext
import os
from tqdm import tqdm

# --- 1. 设置精度和常量 ---
getcontext().prec = 50 # 为金融计算设置高精度
SEC_PER_DAY = 86400
MAX_DURATION_DAYS = 90
PRICE_DECIMALS = Decimal('1e18') # 1e18
LT_DECIMALS = Decimal('10000')  # 10000

# --- 2. 定义所有路径 ---

# 输入：包含 "Reach" 样本的目录
REACH_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\Reach_and_unReach_sample\Reach")

# 输入：包含 ETH 计价价格的目录
PRICE_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\data\every_icon_price_sequence_in_eth")

# (!! 新增 !!) 基础输出目录
BASE_OUTPUT_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\data")

# (!! 新增 !!) 输出 1：存储 HF 波动 CSV 的目录
HF_FLUCTUATION_DIR = BASE_OUTPUT_DIR / "HF_fluctuation_for_samples" / "HF_fluctuation"

# (!! 新增 !!) 输出 2：存储 HF "配方" JSON 的目录
HF_RECIPE_DIR = BASE_OUTPUT_DIR / "HF_fluctuation_for_samples" / "HF_sample_recipe"

# (!! 新增 !!) 输出 3：存储时间范围日志的 CSV
LOG_OUTPUT_FILE = BASE_OUTPUT_DIR / "simulation_log.csv"

# 确保所有输出目录存在
HF_FLUCTUATION_DIR.mkdir(parents=True, exist_ok=True)
HF_RECIPE_DIR.mkdir(parents=True, exist_ok=True)
print(f"HF 时间序列 CSV 输出目录: {HF_FLUCTUATION_DIR}")
print(f"HF 配方 JSON 输出目录: {HF_RECIPE_DIR}")
print(f"日志文件将保存到: {LOG_OUTPUT_FILE}")

# --- 3. 定义核心处理函数 ---

def load_price_data(symbols_needed, price_dir, hourly_index):
    """
    加载所需币种的价格数据, 并将其对齐到统一的每小时索引。
    (此函数与上一版本相同)
    """
    df_merged_prices = pd.DataFrame(index=hourly_index)
    
    for symbol in symbols_needed:
        price_file = price_dir / f"{symbol}_price_in_eth.csv"
        if not price_file.exists():
            print(f"  -> 警告: 找不到价格文件 {price_file}。该资产将被视为价格为 0。")
            df_merged_prices[f"{symbol}_price_eth"] = 0.0 # 价格为 0
            continue
            
        df_price = pd.read_csv(price_file)
        if df_price.empty:
             df_merged_prices[f"{symbol}_price_eth"] = pd.NA
             continue

        df_price['datetime_utc'] = pd.to_datetime(df_price['datetime_utc'], format='ISO8601')
        df_price['datetime_hourly'] = df_price['datetime_utc'].dt.floor('h') # 'h'
        
        df_hourly = df_price.groupby('datetime_hourly').agg(
            price_in_eth=('price_in_eth', 'mean')
        ).reset_index()
        
        df_hourly = df_hourly.set_index('datetime_hourly')
        df_reindexed = df_hourly.reindex(hourly_index, method=None) 
        df_reindexed['price_in_eth'] = df_reindexed['price_in_eth'].interpolate(
            method='time', limit_direction='both'
        )
        df_merged_prices[f"{symbol}_price_eth"] = df_reindexed['price_in_eth']

    df_merged_prices = df_merged_prices.fillna(0.0) # 填充所有剩余的 NaNs
    return df_merged_prices


def simulate_hf_for_sample(sample_json_path, price_dir, timeseries_output_dir, recipe_output_dir):
    """
    为单个样本 JSON 文件计算 HF 波动。
    (!! 已更新 !!)
    """
    
    # --- a. 加载样本并解析 "配方" (Recipe) ---
    tqdm.write(f"--- 正在处理: {sample_json_path.name} ---")
    with open(sample_json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    tx_hash = data['txHash']
    
    try:
        HF_TRUE_ONCHAIN = Decimal(data['health_factor'])
    except (KeyError, TypeError):
        tqdm.write(f"  -> 警告: 样本 JSON 缺少 'health_factor'。无法归一化。跳过。")
        return None

    snapshot = data.get('pre_liquidation_snapshot', [])
    
    # (!! 新增 !!) 存储配方信息
    collaterals = [] # (symbol, amount_Decimal, lt_Decimal)
    collateral_recipes = [] # (用于 JSON 输出)
    debts = []       # (symbol, amount_Decimal)
    debt_recipes = [] # (用于 JSON 输出)
    symbols_needed = set() 
    
    for asset in snapshot:
        symbol = asset['reserve']['symbol']
        decimals = int(asset['reserve']['decimals'])
        
        if asset['usageAsCollateralEnabledOnUser']:
            balance_raw = Decimal(asset['currentATokenBalance'])
            if balance_raw > 0:
                balance = balance_raw / (Decimal(10) ** decimals)
                lt_factor = Decimal(asset['reserve']['reserveLiquidationThreshold']) / LT_DECIMALS
                collaterals.append((symbol, balance, lt_factor))
                collateral_recipes.append({
                    "symbol": symbol,
                    "amount_raw": str(balance_raw),
                    "decimals": decimals,
                    "liquidation_threshold": f"{lt_factor:.4f}"
                })
                symbols_needed.add(symbol)
        
        debt_raw = Decimal(asset['currentTotalDebt'])
        if debt_raw > 0:
            debt = debt_raw / (Decimal(10) ** decimals)
            debts.append((symbol, debt))
            debt_recipes.append({
                "symbol": symbol,
                "amount_raw": str(debt_raw),
                "decimals": decimals
            })
            symbols_needed.add(symbol)

    if not collaterals or not debts:
        tqdm.write("  -> 警告: 样本缺少抵押品或债务数据。无法计算 HF。跳过。")
        return None 

    # --- b. 确定时间范围 (应用 90 天规则) ---
    end_ts = float(data['liquidation_timestamp'])
    start_ts_raw = float(data['last_action_timestamp'])
    duration_sec = end_ts - start_ts_raw
    duration_days = duration_sec / SEC_PER_DAY
    
    if duration_days > MAX_DURATION_DAYS:
        start_ts = end_ts - (MAX_DURATION_DAYS * SEC_PER_DAY)
        truncated = True
    else:
        start_ts = start_ts_raw
        truncated = False

    start_dt_hourly = datetime.fromtimestamp(start_ts, timezone.utc).replace(minute=0, second=0, microsecond=0)
    end_dt_hourly = datetime.fromtimestamp(end_ts, timezone.utc).replace(minute=0, second=0, microsecond=0)
    hourly_index = pd.date_range(start=start_dt_hourly, end=end_dt_hourly, freq='h') # 'h'
    
    if hourly_index.empty:
        tqdm.write("  -> 警告: 计算出的时间范围为空。跳过。")
        return None
        
    # --- c. 加载并对齐价格数据 ---
    df_prices = load_price_data(symbols_needed, price_dir, hourly_index)
    
    # --- d. 向量化计算 *未调整的* HF ---
    total_numerator_series = pd.Series(0.0, index=hourly_index)
    total_denominator_series = pd.Series(0.0, index=hourly_index)
    
    for symbol, amount, lt_factor in collaterals:
        price_series = df_prices[f"{symbol}_price_eth"]
        total_numerator_series += (price_series.astype(float) * float(amount) * float(lt_factor))
        
    for symbol, amount in debts:
        price_series = df_prices[f"{symbol}_price_eth"]
        total_denominator_series += (price_series.astype(float) * float(amount))
    
    df_hf = pd.DataFrame(index=hourly_index)
    df_hf['numerator_eth'] = total_numerator_series
    df_hf['denominator_eth'] = total_denominator_series
    df_hf['health_factor_unadjusted'] = (
        df_hf['numerator_eth'] / df_hf['denominator_eth'].replace(0, pd.NA)
    ).fillna(0.0).astype(float)

    # --- e. 归一化/调整 HF ---
    hf_simulated_final = Decimal(df_hf['health_factor_unadjusted'].iloc[-1])
    
    if hf_simulated_final == 0:
        tqdm.write("  -> 警告: 模拟的最后一个 HF 为 0。无法计算调整因子。")
        GAF = Decimal(1.0) # 调整因子为 1 (即不调整)
    else:
        GAF = HF_TRUE_ONCHAIN / hf_simulated_final

    df_hf['health_factor'] = (df_hf['health_factor_unadjusted'] * float(GAF)).astype(float)
    df_hf.iloc[-1, df_hf.columns.get_loc('health_factor')] = float(HF_TRUE_ONCHAIN)

    # --- f. (!! 新 !!) 保存 HF 时间序列 CSV ---
    output_csv_path = timeseries_output_dir / f"{tx_hash}.csv"
    absolute_path_str = str(output_csv_path.resolve())
    long_path_prefixed = f"\\\\?\\{absolute_path_str}"
    
    # 我们只保存最终的、干净的数据
    df_to_save_csv = df_hf[['health_factor']].copy()
    df_to_save_csv.reset_index(names='datetime_utc').to_csv(long_path_prefixed, index=False, encoding='utf-8')
    tqdm.write(f"  -> (1/2) 成功: HF 时间序列已保存到: {output_csv_path.name}")

    # --- g. (!! 新 !!) 保存 HF 配方 JSON ---
    recipe_data = {
        "txHash": tx_hash,
        "user_id": data['user_id'],
        "simulation_time_range_utc": {
            "start_utc": start_dt_hourly.isoformat(),
            "end_utc": end_dt_hourly.isoformat(),
            "duration_days": (end_ts - start_ts) / SEC_PER_DAY,
            "is_truncated_90d": truncated
        },
        "health_factor_anchors": {
            "HF_True_OnChain": float(HF_TRUE_ONCHAIN),
            "HF_Simulated_Final_Unadjusted": float(hf_simulated_final),
            "Adjustment_Factor_GAF": float(GAF)
        },
        "numerator_collaterals": collateral_recipes,
        "denominator_debts": debt_recipes
    }
    
    output_json_path = recipe_output_dir / f"{tx_hash}.json"
    absolute_json_path_str = str(output_json_path.resolve())
    long_json_path_prefixed = f"\\\\?\\{absolute_json_path_str}"

    with open(long_json_path_prefixed, 'w', encoding='utf-8') as f:
        json.dump(recipe_data, f, indent=4, ensure_ascii=False)
    tqdm.write(f"  -> (2/2) 成功: HF 配方已保存到: {output_json_path.name}")

    # --- h. 返回日志信息 ---
    log_entry = {
        "txHash": tx_hash,
        "status": "Success",
        "output_csv_file": output_csv_path.name,
        "output_json_file": output_json_path.name,
        "is_truncated_90d": truncated,
        "Adjustment_Factor": float(GAF)
    }
    return log_entry

# --- 4. (!! 新 !!) 主执行：循环运行所有样本 ---
print("="*50)
print(f"--- 开始处理 'Reach' 目录中的所有 {len(os.listdir(REACH_DIR))} 个样本 ---")

# 查找所有 Reach 样本
json_files_to_process = [REACH_DIR / f for f in os.listdir(REACH_DIR) if f.endswith('.json')]
simulation_logs = []

# (使用 tqdm 循环)
for sample_path in tqdm(json_files_to_process, desc="模拟所有样本"):
    try:
        log = simulate_hf_for_sample(
            sample_path, 
            PRICE_DIR, 
            HF_FLUCTUATION_DIR, # (!! 新 !!)
            HF_RECIPE_DIR       # (!! 新 !!)
        )
        if log:
            simulation_logs.append(log)
        else:
            simulation_logs.append({
                "txHash": sample_path.stem, "status": "Failed (No HF)", 
                "output_csv_file": None, "output_json_file": None,
                "is_truncated_90d": None, "Adjustment_Factor": None
            })
            
    except Exception as e:
        tqdm.write(f"!! 处理 {sample_path.name} 时发生严重错误: {e}")
        simulation_logs.append({
            "txHash": sample_path.stem, "status": f"Error: {e}",
            "output_csv_file": None, "output_json_file": None,
            "is_truncated_90d": None, "Adjustment_Factor": None
        })

# --- 5. 保存日志文件 ---
if simulation_logs:
    df_log = pd.DataFrame(simulation_logs)
    
    # (修复 Windows MAX_PATH 错误)
    absolute_log_path_str = str(LOG_OUTPUT_FILE.resolve())
    long_log_path_prefixed = f"\\\\?\\{absolute_log_path_str}"
    
    df_log.to_csv(long_log_path_prefixed, index=False, encoding='utf-8')
    print(f"\n--- 模拟日志已保存到: {LOG_OUTPUT_FILE} ---")
else:
    print("\n--- 未处理任何样本，日志未生成 ---")

HF 时间序列 CSV 输出目录: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\data\HF_fluctuation_for_samples\HF_fluctuation
HF 配方 JSON 输出目录: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\data\HF_fluctuation_for_samples\HF_sample_recipe
日志文件将保存到: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\data\simulation_log.csv
--- 开始处理 'Reach' 目录中的所有 16 个样本 ---


模拟所有样本:   0%|          | 0/16 [00:00<?, ?it/s]      

--- 正在处理: 0x5080e4df6da8a0f4f0b0d4fd7e92c7b4b3a37721a9f07516c9e14cdbdacba885.json ---


模拟所有样本:  12%|█▎        | 2/16 [00:00<00:01, 11.47it/s]      

  -> (1/2) 成功: HF 时间序列已保存到: 0x5080e4df6da8a0f4f0b0d4fd7e92c7b4b3a37721a9f07516c9e14cdbdacba885.csv
  -> (2/2) 成功: HF 配方已保存到: 0x5080e4df6da8a0f4f0b0d4fd7e92c7b4b3a37721a9f07516c9e14cdbdacba885.json
--- 正在处理: 0xa0a5a91430df9443b4565ed44c3d21afc1ae142d74dc5fbfd86748db8d708299.json ---
  -> (1/2) 成功: HF 时间序列已保存到: 0xa0a5a91430df9443b4565ed44c3d21afc1ae142d74dc5fbfd86748db8d708299.csv
  -> (2/2) 成功: HF 配方已保存到: 0xa0a5a91430df9443b4565ed44c3d21afc1ae142d74dc5fbfd86748db8d708299.json
--- 正在处理: 0xebad908eac66c1c8e5f12edce2bf827abb10cc34b39f4adc8a5c410ae4dd8b95.json ---


...

  -> (1/2) 成功: HF 时间序列已保存到: 0x1b2546705f783b5dc3c1afe66bc80722f22500a9a5987dc3b466ad38078bcef7.csv
  -> (2/2) 成功: HF 配方已保存到: 0x1b2546705f783b5dc3c1afe66bc80722f22500a9a5987dc3b466ad38078bcef7.json
--- 正在处理: 0xf6d1205710591dfc39ebf1d67313040e40172e1afc01b377d728d1199c43d97b.json ---
  -> (1/2) 成功: HF 时间序列已保存到: 0xf6d1205710591dfc39ebf1d67313040e40172e1afc01b377d728d1199c43d97b.csv
  -> (2/2) 成功: HF 配方已保存到: 0xf6d1205710591dfc39ebf1d67313040e40172e1afc01b377d728d1199c43d97b.json
--- 正在处理: 0xc92e6ec0773de3bc6b5df9c6a633670056cda38fa62b781d83548e5bcb712190.json ---


模拟所有样本:  75%|███████▌  | 12/16 [00:01<00:00, 12.47it/s]      

  -> (1/2) 成功: HF 时间序列已保存到: 0xc92e6ec0773de3bc6b5df9c6a633670056cda38fa62b781d83548e5bcb712190.csv
  -> (2/2) 成功: HF 配方已保存到: 0xc92e6ec0773de3bc6b5df9c6a633670056cda38fa62b781d83548e5bcb712190.json
--- 正在处理: 0x1987831f61cdab421877c4462a5731dd76d3cf9af2040e42b0f19c56ebc0fc27.json ---
  -> (1/2) 成功: HF 时间序列已保存到: 0x1987831f61cdab421877c4462a5731dd76d3cf9af2040e42b0f19c56ebc0fc27.csv
  -> (2/2) 成功: HF 配方已保存到: 0x1987831f61cdab421877c4462a5731dd76d3cf9af2040e42b0f19c56ebc0fc27.json
--- 正在处理: 0x014240c0a39e2721dcf4d10bb5694fb945553c933c5ae948a47ebdb6a8149f2e.json ---
  -> (1/2) 成功: HF 时间序列已保存到: 0x014240c0a39e2721dcf4d10bb5694fb945553c933c5ae948a47ebdb6a8149f2e.csv
  -> (2/2) 成功: HF 配方已保存到: 0x014240c0a39e2721dcf4d10bb5694fb945553c933c5ae948a47ebdb6a8149f2e.json
--- 正在处理: 0x0e1748aa5adba8807a5a1a6c3a4cdfc4fc7df7b9dda57430d6f65e0e3f7f1a84.json ---


模拟所有样本: 100%|██████████| 16/16 [00:01<00:00, 13.03it/s]      

  -> (1/2) 成功: HF 时间序列已保存到: 0x0e1748aa5adba8807a5a1a6c3a4cdfc4fc7df7b9dda57430d6f65e0e3f7f1a84.csv
  -> (2/2) 成功: HF 配方已保存到: 0x0e1748aa5adba8807a5a1a6c3a4cdfc4fc7df7b9dda57430d6f65e0e3f7f1a84.json
--- 正在处理: 0x71227f368cfab933f5e23dc43dda62fc49e3d0b0836cbacb2fee4146d6003e7e.json ---
  -> (1/2) 成功: HF 时间序列已保存到: 0x71227f368cfab933f5e23dc43dda62fc49e3d0b0836cbacb2fee4146d6003e7e.csv
  -> (2/2) 成功: HF 配方已保存到: 0x71227f368cfab933f5e23dc43dda62fc49e3d0b0836cbacb2fee4146d6003e7e.json
--- 正在处理: 0x6f142ea37d6ca73a96d018c5f36b2dac3b7f9dbff45e72be052c7b45e3d86d6b.json ---
  -> (1/2) 成功: HF 时间序列已保存到: 0x6f142ea37d6ca73a96d018c5f36b2dac3b7f9dbff45e72be052c7b45e3d86d6b.csv
  -> (2/2) 成功: HF 配方已保存到: 0x6f142ea37d6ca73a96d018c5f36b2dac3b7f9dbff45e72be052c7b45e3d86d6b.json

--- 模拟日志已保存到: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\data\simulation_log.csv ---


添加详细的json记录每个计算

In [8]:
import pandas as pd
import json
from pathlib import Path
from datetime import datetime, timezone
from decimal import Decimal, getcontext
import os
from tqdm import tqdm

# --- 1. 设置精度和常量 ---
getcontext().prec = 50 
SEC_PER_DAY = 86400
MAX_DURATION_DAYS = 90
PRICE_DECIMALS = Decimal('1e18') 
LT_DECIMALS = Decimal('10000')  

# --- 2. 定义所有路径 ---

# 输入：包含 "Reach" 样本的目录
REACH_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\01_reserve_capture\data\Reach_and_unReach_sample\Reach")

# 输入：包含 ETH 计价价格的目录
PRICE_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\data\every_icon_price_sequence_in_eth")

# (!! 已更新 !!) 基础输出目录
BASE_OUTPUT_DIR = Path(r"F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\data")

# (!! 已更新 !!) 输出 1：存储 HF 波动 CSV 的目录
HF_FLUCTUATION_DIR = BASE_OUTPUT_DIR / "HF_fluctuation_for_samples" / "HF_fluctuation"

# (!! 已更新 !!) 输出 2：存储 HF "配方" JSON 的目录
HF_RECIPE_DIR = BASE_OUTPUT_DIR / "HF_fluctuation_for_samples" / "HF_sample_recipe"

# (!! 新增 !!) 输出 3：存储详细计算过程的 JSON 目录
HF_FLUCTUATION_JSON_DIR = BASE_OUTPUT_DIR / "HF_fluctuation_for_samples" / "HF_fluctuation_json"

# (!! 已更新 !!) 输出 4：存储时间范围日志的 CSV
LOG_OUTPUT_FILE = BASE_OUTPUT_DIR / "simulation_log.csv"

# 确保所有输出目录存在
HF_FLUCTUATION_DIR.mkdir(parents=True, exist_ok=True)
HF_RECIPE_DIR.mkdir(parents=True, exist_ok=True)
HF_FLUCTUATION_JSON_DIR.mkdir(parents=True, exist_ok=True) # (!! 新增 !!)
print(f"HF 时间序列 CSV 输出目录: {HF_FLUCTUATION_DIR}")
print(f"HF 配方 JSON 输出目录: {HF_RECIPE_DIR}")
print(f"HF 详细 JSON 输出目录: {HF_FLUCTUATION_JSON_DIR}") # (!! 新增 !!)
print(f"日志文件将保存到: {LOG_OUTPUT_FILE}")

# --- 3. 定义核心处理函数 ---

def load_price_data(symbols_needed, price_dir, hourly_index):
    """
    加载所需币种的价格数据, 并将其对齐到统一的每小时索引。
    (此函数与上一版本相同)
    """
    df_merged_prices = pd.DataFrame(index=hourly_index)
    
    for symbol in symbols_needed:
        price_file = price_dir / f"{symbol}_price_in_eth.csv"
        if not price_file.exists():
            print(f"  -> 警告: 找不到价格文件 {price_file}。该资产将被视为价格为 0。")
            df_merged_prices[f"{symbol}_price_eth"] = 0.0 
            continue
            
        df_price = pd.read_csv(price_file)
        if df_price.empty:
             df_merged_prices[f"{symbol}_price_eth"] = pd.NA
             continue

        df_price['datetime_utc'] = pd.to_datetime(df_price['datetime_utc'], format='ISO8601')
        df_price['datetime_hourly'] = df_price['datetime_utc'].dt.floor('h') 
        
        df_hourly = df_price.groupby('datetime_hourly').agg(
            price_in_eth=('price_in_eth', 'mean')
        ).reset_index()
        
        df_hourly = df_hourly.set_index('datetime_hourly')
        df_reindexed = df_hourly.reindex(hourly_index, method=None) 
        df_reindexed['price_in_eth'] = df_reindexed['price_in_eth'].interpolate(
            method='time', limit_direction='both'
        )
        df_merged_prices[f"{symbol}_price_eth"] = df_reindexed['price_in_eth']

    df_merged_prices = df_merged_prices.fillna(0.0) 
    return df_merged_prices


def simulate_hf_for_sample(sample_json_path, price_dir, timeseries_output_dir, recipe_output_dir, details_json_output_dir):
    """
    为单个样本 JSON 文件计算 HF 波动。
    (!! 已更新以保存 3 个文件 !!)
    """
    
    # --- a. 加载样本并解析 "配方" (Recipe) ---
    tqdm.write(f"--- 正在处理: {sample_json_path.name} ---")
    with open(sample_json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    tx_hash = data['txHash']
    
    try:
        HF_TRUE_ONCHAIN = Decimal(data['health_factor'])
    except (KeyError, TypeError):
        tqdm.write(f"  -> 警告: 样本 JSON 缺少 'health_factor'。无法归一化。跳过。")
        return None

    snapshot = data.get('pre_liquidation_snapshot', [])
    
    collaterals = [] # (symbol, amount_Decimal, lt_Decimal)
    collateral_recipes = [] # (用于 JSON 输出)
    debts = []       # (symbol, amount_Decimal)
    debt_recipes = [] # (用于 JSON 输出)
    symbols_needed = set() 
    
    for asset in snapshot:
        symbol = asset['reserve']['symbol']
        decimals = int(asset['reserve']['decimals'])
        
        if asset['usageAsCollateralEnabledOnUser']:
            balance_raw = Decimal(asset['currentATokenBalance'])
            if balance_raw > 0:
                balance = balance_raw / (Decimal(10) ** decimals)
                lt_factor = Decimal(asset['reserve']['reserveLiquidationThreshold']) / LT_DECIMALS
                collaterals.append((symbol, balance, lt_factor))
                collateral_recipes.append({
                    "symbol": symbol,
                    "amount_raw": str(balance_raw),
                    "decimals": decimals,
                    "liquidation_threshold": f"{lt_factor:.4f}"
                })
                symbols_needed.add(symbol)
        
        debt_raw = Decimal(asset['currentTotalDebt'])
        if debt_raw > 0:
            debt = debt_raw / (Decimal(10) ** decimals)
            debts.append((symbol, debt))
            debt_recipes.append({
                "symbol": symbol,
                "amount_raw": str(debt_raw),
                "decimals": decimals
            })
            symbols_needed.add(symbol)

    if not collaterals or not debts:
        tqdm.write("  -> 警告: 样本缺少抵押品或债务数据。无法计算 HF。跳过。")
        return None 

    # --- b. 确定时间范围 (应用 90 天规则) ---
    end_ts = float(data['liquidation_timestamp'])
    start_ts_raw = float(data['last_action_timestamp'])
    duration_sec = end_ts - start_ts_raw
    duration_days = duration_sec / SEC_PER_DAY
    
    if duration_days > MAX_DURATION_DAYS:
        start_ts = end_ts - (MAX_DURATION_DAYS * SEC_PER_DAY)
        truncated = True
    else:
        start_ts = start_ts_raw
        truncated = False

    start_dt_hourly = datetime.fromtimestamp(start_ts, timezone.utc).replace(minute=0, second=0, microsecond=0)
    end_dt_hourly = datetime.fromtimestamp(end_ts, timezone.utc).replace(minute=0, second=0, microsecond=0)
    hourly_index = pd.date_range(start=start_dt_hourly, end=end_dt_hourly, freq='h') # 'h'
    
    if hourly_index.empty:
        tqdm.write("  -> 警告: 计算出的时间范围为空。跳过。")
        return None
        
    # --- c. 加载并对齐价格数据 ---
    df_prices = load_price_data(symbols_needed, price_dir, hourly_index)
    
    # --- d. 向量化计算 *未调整的* HF ---
    total_numerator_series = pd.Series(0.0, index=hourly_index)
    total_denominator_series = pd.Series(0.0, index=hourly_index)
    
    for symbol, amount, lt_factor in collaterals:
        price_series = df_prices[f"{symbol}_price_eth"]
        total_numerator_series += (price_series.astype(float) * float(amount) * float(lt_factor))
        
    for symbol, amount in debts:
        price_series = df_prices[f"{symbol}_price_eth"]
        total_denominator_series += (price_series.astype(float) * float(amount))
    
    df_hf = pd.DataFrame(index=hourly_index)
    df_hf['numerator_eth'] = total_numerator_series
    df_hf['denominator_eth'] = total_denominator_series
    df_hf['health_factor_unadjusted'] = (
        df_hf['numerator_eth'] / df_hf['denominator_eth'].replace(0, pd.NA)
    ).fillna(0.0).astype(float)

    # --- e. 归一化/调整 HF ---
    hf_simulated_final = Decimal(df_hf['health_factor_unadjusted'].iloc[-1])
    
    if hf_simulated_final == 0:
        tqdm.write("  -> 警告: 模拟的最后一个 HF 为 0。无法计算调整因子。")
        GAF = Decimal(1.0) 
    else:
        GAF = HF_TRUE_ONCHAIN / hf_simulated_final

    df_hf['health_factor'] = (df_hf['health_factor_unadjusted'] * float(GAF)).astype(float)
    df_hf.iloc[-1, df_hf.columns.get_loc('health_factor')] = float(HF_TRUE_ONCHAIN)

    # --- f. 保存 HF 时间序列 CSV (最终的 HF) ---
    output_csv_path = timeseries_output_dir / f"{tx_hash}.csv"
    absolute_path_str = str(output_csv_path.resolve())
    long_path_prefixed = f"\\\\?\\{absolute_path_str}"
    
    df_to_save_csv = df_hf[['health_factor']].copy()
    df_to_save_csv.reset_index(names='datetime_utc').to_csv(long_path_prefixed, index=False, encoding='utf-8')
    tqdm.write(f"  -> (1/3) 成功: HF 时间序列已保存到: {output_csv_path.name}")

    # --- g. 保存 HF 配方 JSON (静态信息) ---
    recipe_data = {
        "txHash": tx_hash,
        "user_id": data['user_id'],
        "simulation_time_range_utc": {
            "start_utc": start_dt_hourly.isoformat(),
            "end_utc": end_dt_hourly.isoformat(),
            "duration_days": (end_ts - start_ts) / SEC_PER_DAY,
            "is_truncated_90d": truncated
        },
        "health_factor_anchors": {
            "HF_True_OnChain": float(HF_TRUE_ONCHAIN),
            "HF_Simulated_Final_Unadjusted": float(hf_simulated_final),
            "Adjustment_Factor_GAF": float(GAF)
        },
        "numerator_collaterals": collateral_recipes,
        "denominator_debts": debt_recipes
    }
    
    output_json_path = recipe_output_dir / f"{tx_hash}.json"
    absolute_json_path_str = str(output_json_path.resolve())
    long_json_path_prefixed = f"\\\\?\\{absolute_json_path_str}"

    with open(long_json_path_prefixed, 'w', encoding='utf-8') as f:
        json.dump(recipe_data, f, indent=4, ensure_ascii=False)
    tqdm.write(f"  -> (2/3) 成功: HF 配方已保存到: {output_json_path.name}")

    # --- h. (!! 新增 !!) 保存详细计算过程 JSON (动态信息) ---
    output_details_json_path = details_json_output_dir / f"{tx_hash}_details.json"
    absolute_details_path_str = str(output_details_json_path.resolve())
    long_details_path_prefixed = f"\\\\?\\{absolute_details_path_str}"
    
    hourly_json_output = []
    
    # 遍历我们计算的 DataFrame 的每一行
    for timestamp, row in df_hf.iterrows():
        hourly_data = {
            "datetime_utc": timestamp.isoformat(),
            "numerator_eth": row['numerator_eth'],
            "denominator_eth": row['denominator_eth'],
            "health_factor_unadjusted": row['health_factor_unadjusted'],
            "health_factor_adjusted": row['health_factor'],
            "price_details_eth": {}
        }
        
        # 从 df_prices 中获取该时间戳的价格
        for symbol in symbols_needed:
            hourly_data["price_details_eth"][symbol] = df_prices.at[timestamp, f"{symbol}_price_eth"]
            
        hourly_json_output.append(hourly_data)

    with open(long_details_path_prefixed, 'w', encoding='utf-8') as f:
        json.dump(hourly_json_output, f, indent=4, ensure_ascii=False)
    tqdm.write(f"  -> (3/3) 成功: 详细计算 JSON 已保存到: {output_details_json_path.name}")

    # --- i. 返回日志信息 ---
    log_entry = {
        "txHash": tx_hash,
        "status": "Success",
        "output_csv_file": output_csv_path.name,
        "output_recipe_json_file": output_json_path.name,
        "output_details_json_file": output_details_json_path.name,
        "is_truncated_90d": truncated,
        "Adjustment_Factor": float(GAF)
    }
    return log_entry

# --- 4. 主执行：循环运行所有样本 ---
print("="*50)
print(f"--- 开始处理 'Reach' 目录中的所有 {len(os.listdir(REACH_DIR))} 个样本 ---")

# 查找所有 Reach 样本
json_files_to_process = [REACH_DIR / f for f in os.listdir(REACH_DIR) if f.endswith('.json')]
simulation_logs = []

# (使用 tqdm 循环)
for sample_path in tqdm(json_files_to_process, desc="模拟所有样本"):
    try:
        log = simulate_hf_for_sample(
            sample_path, 
            PRICE_DIR, 
            HF_FLUCTUATION_DIR,       # (!! 已更新 !!)
            HF_RECIPE_DIR,            # (!! 已更新 !!)
            HF_FLUCTUATION_JSON_DIR   # (!! 新增 !!)
        )
        if log:
            simulation_logs.append(log)
        else:
            simulation_logs.append({
                "txHash": sample_path.stem, "status": "Failed (No HF)", 
                "output_csv_file": None, "output_recipe_json_file": None, "output_details_json_file": None,
                "is_truncated_90d": None, "Adjustment_Factor": None
            })
            
    except Exception as e:
        tqdm.write(f"!! 处理 {sample_path.name} 时发生严重错误: {e}")
        simulation_logs.append({
            "txHash": sample_path.stem, "status": f"Error: {e}",
            "output_csv_file": None, "output_recipe_json_file": None, "output_details_json_file": None,
            "is_truncated_90d": None, "Adjustment_Factor": None
        })

# --- 5. 保存日志文件 ---
if simulation_logs:
    df_log = pd.DataFrame(simulation_logs)
    
    # (修复 Windows MAX_PATH 错误)
    absolute_log_path_str = str(LOG_OUTPUT_FILE.resolve())
    long_log_path_prefixed = f"\\\\?\\{absolute_log_path_str}"
    
    df_log.to_csv(long_log_path_prefixed, index=False, encoding='utf-8')
    print(f"\n--- 模拟日志已保存到: {LOG_OUTPUT_FILE} ---")
else:
    print("\n--- 未处理任何样本，日志未生成 ---")

HF 时间序列 CSV 输出目录: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\data\HF_fluctuation_for_samples\HF_fluctuation
HF 配方 JSON 输出目录: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\data\HF_fluctuation_for_samples\HF_sample_recipe
HF 详细 JSON 输出目录: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\data\HF_fluctuation_for_samples\HF_fluctuation_json
日志文件将保存到: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\data\simulation_log.csv
--- 开始处理 'Reach' 目录中的所有 16 个样本 ---


模拟所有样本:   0%|          | 0/16 [00:00<?, ?it/s]      

--- 正在处理: 0x5080e4df6da8a0f4f0b0d4fd7e92c7b4b3a37721a9f07516c9e14cdbdacba885.json ---
  -> (1/3) 成功: HF 时间序列已保存到: 0x5080e4df6da8a0f4f0b0d4fd7e92c7b4b3a37721a9f07516c9e14cdbdacba885.csv
  -> (2/3) 成功: HF 配方已保存到: 0x5080e4df6da8a0f4f0b0d4fd7e92c7b4b3a37721a9f07516c9e14cdbdacba885.json
  -> (3/3) 成功: 详细计算 JSON 已保存到: 0x5080e4df6da8a0f4f0b0d4fd7e92c7b4b3a37721a9f07516c9e14cdbdacba885_details.json
--- 正在处理: 0xa0a5a91430df9443b4565ed44c3d21afc1ae142d74dc5fbfd86748db8d708299.json ---
  -> (1/3) 成功: HF 时间序列已保存到: 0xa0a5a91430df9443b4565ed44c3d21afc1ae142d74dc5fbfd86748db8d708299.csv
  -> (2/3) 成功: HF 配方已保存到: 0xa0a5a91430df9443b4565ed44c3d21afc1ae142d74dc5fbfd86748db8d708299.json


模拟所有样本:  19%|█▉        | 3/16 [00:00<00:01,  8.46it/s]      

  -> (3/3) 成功: 详细计算 JSON 已保存到: 0xa0a5a91430df9443b4565ed44c3d21afc1ae142d74dc5fbfd86748db8d708299_details.json
--- 正在处理: 0xebad908eac66c1c8e5f12edce2bf827abb10cc34b39f4adc8a5c410ae4dd8b95.json ---
  -> (1/3) 成功: HF 时间序列已保存到: 0xebad908eac66c1c8e5f12edce2bf827abb10cc34b39f4adc8a5c410ae4dd8b95.csv
  -> (2/3) 成功: HF 配方已保存到: 0xebad908eac66c1c8e5f12edce2bf827abb10cc34b39f4adc8a5c410ae4dd8b95.json
  -> (3/3) 成功: 详细计算 JSON 已保存到: 0xebad908eac66c1c8e5f12edce2bf827abb10cc34b39f4adc8a5c410ae4dd8b95_details.json
--- 正在处理: 0xa4871addc95f07db81844be1937fa76b6e01e32bf093bada8b6df3ad6321f655.json ---


...

  -> (2/3) 成功: HF 配方已保存到: 0x1987831f61cdab421877c4462a5731dd76d3cf9af2040e42b0f19c56ebc0fc27.json
  -> (3/3) 成功: 详细计算 JSON 已保存到: 0x1987831f61cdab421877c4462a5731dd76d3cf9af2040e42b0f19c56ebc0fc27_details.json
--- 正在处理: 0x014240c0a39e2721dcf4d10bb5694fb945553c933c5ae948a47ebdb6a8149f2e.json ---
  -> (1/3) 成功: HF 时间序列已保存到: 0x014240c0a39e2721dcf4d10bb5694fb945553c933c5ae948a47ebdb6a8149f2e.csv
  -> (2/3) 成功: HF 配方已保存到: 0x014240c0a39e2721dcf4d10bb5694fb945553c933c5ae948a47ebdb6a8149f2e.json
  -> (3/3) 成功: 详细计算 JSON 已保存到: 0x014240c0a39e2721dcf4d10bb5694fb945553c933c5ae948a47ebdb6a8149f2e_details.json
--- 正在处理: 0x0e1748aa5adba8807a5a1a6c3a4cdfc4fc7df7b9dda57430d6f65e0e3f7f1a84.json ---
  -> (1/3) 成功: HF 时间序列已保存到: 0x0e1748aa5adba8807a5a1a6c3a4cdfc4fc7df7b9dda57430d6f65e0e3f7f1a84.csv
  -> (2/3) 成功: HF 配方已保存到: 0x0e1748aa5adba8807a5a1a6c3a4cdfc4fc7df7b9dda57430d6f65e0e3f7f1a84.json


模拟所有样本:  94%|█████████▍| 15/16 [00:02<00:00,  6.94it/s]      

  -> (3/3) 成功: 详细计算 JSON 已保存到: 0x0e1748aa5adba8807a5a1a6c3a4cdfc4fc7df7b9dda57430d6f65e0e3f7f1a84_details.json
--- 正在处理: 0x71227f368cfab933f5e23dc43dda62fc49e3d0b0836cbacb2fee4146d6003e7e.json ---
  -> (1/3) 成功: HF 时间序列已保存到: 0x71227f368cfab933f5e23dc43dda62fc49e3d0b0836cbacb2fee4146d6003e7e.csv
  -> (2/3) 成功: HF 配方已保存到: 0x71227f368cfab933f5e23dc43dda62fc49e3d0b0836cbacb2fee4146d6003e7e.json
  -> (3/3) 成功: 详细计算 JSON 已保存到: 0x71227f368cfab933f5e23dc43dda62fc49e3d0b0836cbacb2fee4146d6003e7e_details.json
--- 正在处理: 0x6f142ea37d6ca73a96d018c5f36b2dac3b7f9dbff45e72be052c7b45e3d86d6b.json ---


模拟所有样本: 100%|██████████| 16/16 [00:02<00:00,  6.28it/s]      

  -> (1/3) 成功: HF 时间序列已保存到: 0x6f142ea37d6ca73a96d018c5f36b2dac3b7f9dbff45e72be052c7b45e3d86d6b.csv
  -> (2/3) 成功: HF 配方已保存到: 0x6f142ea37d6ca73a96d018c5f36b2dac3b7f9dbff45e72be052c7b45e3d86d6b.json
  -> (3/3) 成功: 详细计算 JSON 已保存到: 0x6f142ea37d6ca73a96d018c5f36b2dac3b7f9dbff45e72be052c7b45e3d86d6b_details.json

--- 模拟日志已保存到: F:\Learning_journal_at_CUHK\FTEC5520_Appl Blockchain & Cryptocur\aave_data_collection\src\02_HF_simulation\02_data_collection_for_HF_simulation\02_simulate_HF_for_every_sample\data\simulation_log.csv ---
